# Story Analyst Blueprint Analyzer

Thống kê blueprint và tính điểm chất lượng qua các lần run.

In [4]:
import json
from pathlib import Path

BLUEPRINT_FILE = 'D:/record_by_me/README/CiB/outputs/story_blueprint.json'

with open(BLUEPRINT_FILE, 'r', encoding='utf-8') as f:
    blueprint = json.load(f)

In [5]:
stats = {}
director = blueprint.get('director_view', {})

stats['main_characters'] = len(director.get('main_characters', []))
stats['main_conflicts'] = len(director.get('main_conflicts', []))
stats['critical_path_summary'] = 1 if director.get('critical_path_summary') else 0
stats['top_hooks'] = len(director.get('top_hooks', []))

story_tree = blueprint.get('story_tree', {})
act_count = sequence_count = scene_count = beat_count = summary_count = 0

def traverse(node):
    global act_count, sequence_count, scene_count, beat_count, summary_count
    if node.get('summary'): summary_count += 1
    t = node.get('type')
    if t == 'act': act_count += 1
    elif t == 'sequence': sequence_count += 1
    elif t == 'scene':
        scene_count += 1
        for beat in node.get('beats', []):
            beat_count += 1
            if beat.get('summary'): summary_count += 1
    for child in node.get('children', []):
        traverse(child)

traverse(story_tree)

stats['acts'] = act_count
stats['sequences'] = sequence_count
stats['scenes'] = scene_count
stats['beats'] = beat_count
stats['summaries'] = summary_count

stats['causality_nodes'] = len(blueprint.get('causality_graph', {}).get('nodes', []))
stats['causality_edges'] = len(blueprint.get('causality_graph', {}).get('edges', []))
stats['relationship_nodes'] = len(blueprint.get('character_relationship_graph', {}).get('nodes', []))
stats['relationship_edges'] = len(blueprint.get('character_relationship_graph', {}).get('edges', []))
stats['relationship_events'] = sum(
    len(edge.get('timeline', []))
    for edge in blueprint.get('character_relationship_graph', {}).get('edges', [])
)

prop_nodes = blueprint.get('asset_and_prop_graph', {}).get('nodes', [])
stats['props'] = len(prop_nodes)
stats['prop_states'] = sum(
    len(node.get('ownership_history', [])) +
    len(node.get('location_history', [])) +
    len(node.get('state_history', []))
    for node in prop_nodes
)

stats['presence_rows'] = len(blueprint.get('presence_matrix', []))
stats['verification_rules'] = len(blueprint.get('reflection_verification_rules', []))

for k,v in stats.items():
    print(f'{k:30}: {v}')

main_characters               : 2
main_conflicts                : 2
critical_path_summary         : 1
top_hooks                     : 3
acts                          : 1
sequences                     : 1
scenes                        : 2
beats                         : 14
summaries                     : 5
causality_nodes               : 16
causality_edges               : 12
relationship_nodes            : 2
relationship_edges            : 1
relationship_events           : 2
props                         : 3
prop_states                   : 10
presence_rows                 : 2
verification_rules            : 2


In [6]:
# Blueprint Quality Score

def clamp(x):
    return max(0.0, min(10.0, float(x)))

director = blueprint.get('director_view', {})

completeness = (
    (2.5 if stats['main_characters'] > 0 else 0) +
    (2.5 if stats['main_conflicts'] > 0 else 0) +
    (2.5 if stats['critical_path_summary'] > 0 else 0) +
    (2.5 if stats['top_hooks'] > 0 else 0)
)

total_nodes = stats['causality_nodes'] + stats['relationship_nodes'] + stats['props']
total_edges = stats['causality_edges'] + stats['relationship_edges']
graph_density = clamp((total_edges / max(total_nodes, 1)) * 10)

compression = blueprint.get('narrative_compression_model', {})
tiers = compression.get('importance_tiers', {})
total_scenes_in_tiers = sum(len(v) for v in tiers.values())
compression_score = clamp(
    (total_scenes_in_tiers / max(stats['scenes'], 1)) * 10
)

hook_score = clamp(stats['top_hooks'] * 3.33)

continuity_score = clamp(stats['verification_rules'] / max(stats['scenes'], 1) * 10)

overall = round((
    completeness +
    graph_density +
    compression_score +
    hook_score +
    continuity_score
) / 5, 2)

print('Completeness Score :', round(completeness, 2))
print('Graph Density Score:', round(graph_density, 2))
print('Compression Score :', round(compression_score, 2))
print('Hook Quality Proxy:', round(hook_score, 2))
print('Continuity Score  :', round(continuity_score, 2))
print('\nOverall Blueprint Score:', overall, '/ 10')

Completeness Score : 10.0
Graph Density Score: 6.19
Compression Score : 10.0
Hook Quality Proxy: 9.99
Continuity Score  : 10.0

Overall Blueprint Score: 9.24 / 10
